In [1]:
import os
import pandas as pd

from utils import (
    ALL_METRICS,
    check_metric_nan_rows,
    check_duplicate_runs,
    filter_existing_trials,
    get_experiment_results,
    get_experiments_metadata,
    get_metric_display_mean,
    plot_experiment_bar,
    plot_experiment_radar,
)

BASE_OUTPUT_DIR = "../labrag"

In [ ]:
check_metric_nan_rows(BASE_OUTPUT_DIR)

In [ ]:
check_duplicate_runs(BASE_OUTPUT_DIR)

In [ ]:
for dataset in [
    "mimic-cxr",
    "chexpertplus",
]:
    for section in [
        "findings",
        "impression",
    ]:
        dataset_section_results = []
        fig_dir = f"../figs/{dataset}-{section}"
        os.makedirs(fig_dir, exist_ok=True)
        experiments = get_experiments_metadata(
            dataset=dataset,
            section=section,
            result_dir=BASE_OUTPUT_DIR,
        )
        for exp_name, (exp_dir, exp_trials) in experiments.items():
            save_name = exp_name.replace(' ', '-').lower()
            available_trials = filter_existing_trials(
                exp_name=f"{dataset}/{section}/{exp_name}",
                exp_dir=exp_dir,
                exp_trials=exp_trials,
            )
            if len(available_trials) < 2:
                print(
                    f"WARNING: skipping {dataset}/{section}/{exp_name}: "
                    f"need at least 2 trials, found {len(available_trials)}"
                )
                continue
            trial_dfs = get_experiment_results(
                exp_dir=exp_dir,
                exp_trials=available_trials,
                # normalize_bertscore_lang="en",
            )

            if dataset == "mimic-cxr":
                title = f"MIMIC-CXR - {section.title()}"
            elif dataset == "chexpertplus":
                title = f"CheXpert Plus - {section.title()}"
            else:
                raise ValueError(f"Please add custom title for dataset {dataset}")
            
            # Bar plots
            fig_bar = plot_experiment_bar(
                title=title,
                exp_name=exp_name,
                exp_trials=available_trials,
                trial_dfs=trial_dfs,
                metrics=ALL_METRICS,
            )
            fig_bar.savefig(os.path.join(fig_dir, f"{save_name}-bar.pdf"))
            fig_bar.savefig(os.path.join(fig_dir, f"{save_name}-bar.png"), dpi=300)

            # Radar plots
            fig_radar = plot_experiment_radar(
                title=title,
                exp_name=exp_name,
                exp_trials=available_trials,
                trial_dfs=trial_dfs,
                metrics=ALL_METRICS,
            )
            fig_radar.savefig(os.path.join(fig_dir, f"{save_name}-radar.pdf"))
            fig_radar.savefig(os.path.join(fig_dir, f"{save_name}-radar.png"), dpi=300)

            # Tabular results
            rows = []
            for trial_df in trial_dfs:
                row = trial_df[ALL_METRICS].mean().copy()
                if "radcliqv1" in row.index:
                    row["radcliqv1"] = get_metric_display_mean("radcliqv1", trial_df)
                rows.append(row)
            df = pd.DataFrame(rows)
            df["experiment"] = exp_name
            df["variable"] = [v for v, f in available_trials]
            df = df[["experiment", "variable"] + ALL_METRICS]
            dataset_section_results.append(df)

        # write results, use other notebook for formatting TeX tables
        if dataset_section_results:
            dataset_section_results = pd.concat(dataset_section_results, ignore_index=True)
            dataset_section_results.to_csv(os.path.join(fig_dir, f"all-results.csv"), index=False)